In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import plotly.express as px
from plotly.offline import init_notebook_mode
import re
import nltk
from nltk.corpus import stopwords
from tqdm import tqdm
from nltk.stem import WordNetLemmatizer
import spacy

nltk.download("omw-1.4", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
tqdm.pandas()
spacy_eng = spacy.load("en_core_web_sm")
lem = WordNetLemmatizer()
init_notebook_mode(connected=True)
sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = (20,8)
plt.rcParams["font.size"] = 18

In [2]:
data = pd.read_csv('train.csv')
data.head(15)

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0
5,5,11,12,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan...",1
6,6,13,14,Should I buy tiago?,What keeps childern active and far from phone ...,0
7,7,15,16,How can I be a good geologist?,What should I do to be a great geologist?,1
8,8,17,18,When do you use シ instead of し?,"When do you use ""&"" instead of ""and""?",0
9,9,19,20,Motorola (company): Can I hack my Charter Moto...,How do I hack Motorola DCX3400 for free internet?,0


In [3]:
data.isnull().sum()

id              0
qid1            0
qid2            0
question1       1
question2       2
is_duplicate    0
dtype: int64

In [4]:
data.dropna(inplace=True)

In [5]:
fig = px.pie(data, values='id', names='is_duplicate', title='Duplicate And Non Duplicate Questions Percentage')
fig.show()

In [6]:
def text_cleaning(words):
    question = re.sub(r'\s+\n+', ' ', words)  # remove extra spaces and newlines
    question = re.sub(r'[^a-zA-Z0-9]', ' ', question)
    question = question.lower()
    return question

In [7]:
data['question1_clean'] = data['question1'].progress_apply(text_cleaning)
data['question2_clean'] = data['question2'].progress_apply(text_cleaning)
data

100%|██████████| 404287/404287 [00:01<00:00, 220265.97it/s]


,id,qid1,qid2,question1,question2,is_duplicate,question1_clean,question2_clean
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0,what is the story of kohinoor koh i noor dia...,what would happen if the indian government sto...
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0,why am i mentally very lonely how can i solve...,find the remainder when math 23 24 math i...
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0,which one dissolve in water quikly sugar salt...,which fish would survive in salt water
...,...,...,...,...,...,...,...,...
404285,404285,433578,379845,How many keywords are there in the Racket prog...,How many keywords are there in PERL Programmin...,0,how many keywords are there in the racket prog...,how many keywords are there in perl programmin...
404286,404286,18840,155606,Do you believe there is life after death?,Is it true that there is life after death?,1,do you believe there is life after death,is it true that there is life after death
404287,404287,537928,537929,What is one coin?,What's this coin?,0,what is one coin,what s this coin
404288,404288,537930,537931,What is the approx annual cost of living while...,I am having little hairfall problem but I want...,0,what is the approx annual cost of living while...,i am having little hairfall problem but i want...


In [8]:
data['question1_length'] = data['question1_clean'].apply(lambda x: len(x.split()))
data['question2_length'] = data['question2_clean'].apply(lambda x: len(x.split()))

In [9]:
px.histogram(data, x='question1_length', color='is_duplicate', title='Question 1 Length Distribution Plot')

In [10]:
px.histogram(data, x='question2_length', color='is_duplicate', title='Question 2 Length Distribution Plot')

In [11]:
questions1 = data['question1_clean'].tolist()
questions2 = data['question2_clean'].tolist()

In [12]:
wordcloud = WordCloud(max_words=1500, background_color='black').generate(" ".join(questions1))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Words From Question1')
plt.axis('off')
plt.show()

In [13]:
wordcloud = WordCloud(max_words=1500, background_color='black').generate(" ".join(questions2))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Words From Question2')
plt.axis('off')
plt.show()

In [14]:
data['question1_length'].describe()

count    404287.000000
mean         11.127968
std           5.571416
min           0.000000
25%           7.000000
50%          10.000000
75%          13.000000
max         128.000000
Name: question1_length, dtype: float64

In [15]:
data['question2_length'].describe()

count    404287.000000
mean         11.376792
std           6.480827
min           0.000000
25%           7.000000
50%          10.000000
75%          13.000000
max         248.000000
Name: question2_length, dtype: float64

In [16]:
import tensorflow as tf
import tf_keras
from tf_keras.preprocessing.text import Tokenizer
from tf_keras.preprocessing.sequence import pad_sequences
from tf_keras.utils import to_categorical
from tf_keras.models import Sequential, Model
from tf_keras import layers
from tf_keras.layers import (Embedding, Layer, Dense, Dropout, MultiHeadAttention,
                              LayerNormalization, Input, GlobalAveragePooling1D,
                              LSTM, Bidirectional)
from tf_keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, DataCollatorWithPadding, TFAutoModel,
                           DistilBertConfig, TFDistilBertModel, BertConfig,
                           TFBertModel, TFRobertaModel)
from datasets import load_dataset

In [17]:
model_checkpoint = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [18]:
def encode_text(text, tokenizer):
    encoded = tokenizer.batch_encode_plus(text, add_special_tokens=True, max_length=50,padding='max_length', truncation=True, return_attention_mask=True, return_tensors='tf')

    input_ids = np.array(encoded['input_ids'], dtype='int32')
    attention_masks = np.array(encoded['attention_mask'], dtype='int32')
    return {'input_ids': input_ids, 'attention_masks': attention_masks}

In [19]:
n_sample = min(400000, len(data))
data = data.sample(n_sample)
train = data.iloc[:int(n_sample * 0.80), :]
val = data.iloc[int(n_sample * 0.80):, :]

question1_train = encode_text(train["question1_clean"].tolist(), tokenizer)
question2_train = encode_text(train["question2_clean"].tolist(), tokenizer)
question1_val = encode_text(val["question1_clean"].tolist(), tokenizer)
question2_val = encode_text(val["question2_clean"].tolist(), tokenizer)

y_train = train["is_duplicate"].values
y_val = val["is_duplicate"].values

In [20]:
# Make data with 3 questions where 3rd question is entire random and not related to other questions

In [21]:
duplicate_data = data.query("is_duplicate == 1").copy()
duplicate_data

,id,qid1,qid2,question1,question2,is_duplicate,question1_clean,question2_clean,question1_length,question2_length
281306,281306,98572,37324,What is the difference between AC and DC curr...,What is the difference between AC current and ...,1,what is the difference between ac and dc curr...,what is the difference between ac current and ...,9,10
247557,247557,29618,29028,What are the best laptops one can go for under...,What is the best laptop under the category of ...,1,what are the best laptops one can go for under...,what is the best laptop under the category of ...,14,11
367006,367006,179590,456245,Which one to buy? Honda hornet 160R or Suzuki ...,I am confused between Suzuki Gixxer SF FI and ...,1,which one to buy honda hornet 160r or suzuki ...,i am confused between suzuki gixxer sf fi and ...,10,26
148706,148706,234401,234402,Which is the best motor bike in the Royal Enfi...,Which is the best royal Enfield bike in 2016?,1,which is the best motor bike in the royal enfi...,which is the best royal enfield bike in 2016,11,9
145300,145300,229764,229765,What are the functional differences between ve...,What is the main difference between veins and ...,1,what are the functional differences between ve...,what is the main difference between veins and ...,9,9
...,...,...,...,...,...,...,...,...,...,...
280256,280256,399809,399810,What does Quora use to automatically update a ...,How does the Quora view counting system work?,1,what does quora use to automatically update a ...,how does the quora view counting system work,12,8
139664,139664,222129,222130,When you can't sleep at night what do you do?,"When you can not sleep at nights, what would y...",1,when you can t sleep at night what do you do,when you can not sleep at nights what would y...,11,11
294251,294251,92545,16340,"What is the Sahara, and how do the average tem...","What is the Sahara, and how do the average tem...",1,what is the sahara and how do the average tem...,what is the sahara and how do the average tem...,19,20
200142,200142,240469,34554,What was your initial reaction on getting to k...,What do you think about Modi Government decisi...,1,what was your initial reaction on getting to k...,what do you think about modi government decisi...,20,13


In [22]:
duplicate_data.shape

(147651, 10)

In [23]:
non_duplicate_data = data.query('is_duplicate == 0')
question3_col = list(non_duplicate_data[:duplicate_data.shape[0]]['question1'])
question3_col

['Why is Puerto Rico not a US state?',
 'How do I make myself mentally and emotionally strong?',
 'Does centrifugal force always acts or only in case of non inertial frame of reference?',
 'What should I do for being a WWE superstar?',
 'Is there real magic in the world?',
 'What is ordinary RC moment resisting frame?',
 'How do I recover deleted iCloud backup?',
 'How do I mirror my PC screen to a Samsung Smart TV wirelessly?',
 'Any engineering students or an engineer who can guide me the scope of mechatronics in abroad plzzz I really need your help?',
 'Art Conservation and Restoration: What are the conditions of replacing a bulb in a Dan Flavin piece?',
 'How do you fix an LG TV turning off all by itself?',
 'How do I understand the time value of money?',
 'Is this English sentence correct in grammar?',
 'Why does she treat me badly? And how can I over come it?',
 'Is it possible to be mentored by Jimmy Wales?',
 "What is Gabriel marcel's critiques on existentialism?",
 'What are s

In [24]:
duplicate_data['question3'] = question3_col

In [25]:
duplicate_data.head(15)

,id,qid1,qid2,question1,question2,is_duplicate,question1_clean,question2_clean,question1_length,question2_length,question3
281306,281306,98572,37324,What is the difference between AC and DC curr...,What is the difference between AC current and ...,1,what is the difference between ac and dc curr...,what is the difference between ac current and ...,9,10,Why is Puerto Rico not a US state?
247557,247557,29618,29028,What are the best laptops one can go for under...,What is the best laptop under the category of ...,1,what are the best laptops one can go for under...,what is the best laptop under the category of ...,14,11,How do I make myself mentally and emotionally ...
367006,367006,179590,456245,Which one to buy? Honda hornet 160R or Suzuki ...,I am confused between Suzuki Gixxer SF FI and ...,1,which one to buy honda hornet 160r or suzuki ...,i am confused between suzuki gixxer sf fi and ...,10,26,Does centrifugal force always acts or only in ...
148706,148706,234401,234402,Which is the best motor bike in the Royal Enfi...,Which is the best royal Enfield bike in 2016?,1,which is the best motor bike in the royal enfi...,which is the best royal enfield bike in 2016,11,9,What should I do for being a WWE superstar?
145300,145300,229764,229765,What are the functional differences between ve...,What is the main difference between veins and ...,1,what are the functional differences between ve...,what is the main difference between veins and ...,9,9,Is there real magic in the world?
19592,19592,37014,37015,Who is the most beautiful living actress?,Who is the most beautiful actress in the world?,1,who is the most beautiful living actress,who is the most beautiful actress in the world,7,9,What is ordinary RC moment resisting frame?
268993,268993,94774,331714,Since light is the fastest thing known in the ...,"If light can't escape a black hole, does this ...",1,since light is the fastest thing known in the ...,if light can t escape a black hole does this ...,33,22,How do I recover deleted iCloud backup?
20207,20207,38146,15970,How do I apply for pan card in bank?,How do I apply for PAN Card?,1,how do i apply for pan card in bank,how do i apply for pan card,9,7,How do I mirror my PC screen to a Samsung Smar...
341561,341561,180752,469419,What are the ill effects of porn?,What are the harmful effects of pornography?,1,what are the ill effects of porn,what are the harmful effects of pornography,7,7,Any engineering students or an engineer who ca...
352716,352716,481664,481665,How do I take a picture of myself?,What are some good ways to take a picture of m...,1,how do i take a picture of myself,what are some good ways to take a picture of m...,8,11,Art Conservation and Restoration: What are the...


In [26]:
duplicate_data['question3_clean'] = duplicate_data['question3'].progress_apply(text_cleaning)

100%|██████████| 147651/147651 [00:00<00:00, 206318.41it/s]


In [27]:
duplicate_data.head(25)

,id,qid1,qid2,question1,question2,is_duplicate,question1_clean,question2_clean,question1_length,question2_length,question3,question3_clean
281306,281306,98572,37324,What is the difference between AC and DC curr...,What is the difference between AC current and ...,1,what is the difference between ac and dc curr...,what is the difference between ac current and ...,9,10,Why is Puerto Rico not a US state?,why is puerto rico not a us state
247557,247557,29618,29028,What are the best laptops one can go for under...,What is the best laptop under the category of ...,1,what are the best laptops one can go for under...,what is the best laptop under the category of ...,14,11,How do I make myself mentally and emotionally ...,how do i make myself mentally and emotionally ...
367006,367006,179590,456245,Which one to buy? Honda hornet 160R or Suzuki ...,I am confused between Suzuki Gixxer SF FI and ...,1,which one to buy honda hornet 160r or suzuki ...,i am confused between suzuki gixxer sf fi and ...,10,26,Does centrifugal force always acts or only in ...,does centrifugal force always acts or only in ...
148706,148706,234401,234402,Which is the best motor bike in the Royal Enfi...,Which is the best royal Enfield bike in 2016?,1,which is the best motor bike in the royal enfi...,which is the best royal enfield bike in 2016,11,9,What should I do for being a WWE superstar?,what should i do for being a wwe superstar
145300,145300,229764,229765,What are the functional differences between ve...,What is the main difference between veins and ...,1,what are the functional differences between ve...,what is the main difference between veins and ...,9,9,Is there real magic in the world?,is there real magic in the world
19592,19592,37014,37015,Who is the most beautiful living actress?,Who is the most beautiful actress in the world?,1,who is the most beautiful living actress,who is the most beautiful actress in the world,7,9,What is ordinary RC moment resisting frame?,what is ordinary rc moment resisting frame
268993,268993,94774,331714,Since light is the fastest thing known in the ...,"If light can't escape a black hole, does this ...",1,since light is the fastest thing known in the ...,if light can t escape a black hole does this ...,33,22,How do I recover deleted iCloud backup?,how do i recover deleted icloud backup
20207,20207,38146,15970,How do I apply for pan card in bank?,How do I apply for PAN Card?,1,how do i apply for pan card in bank,how do i apply for pan card,9,7,How do I mirror my PC screen to a Samsung Smar...,how do i mirror my pc screen to a samsung smar...
341561,341561,180752,469419,What are the ill effects of porn?,What are the harmful effects of pornography?,1,what are the ill effects of porn,what are the harmful effects of pornography,7,7,Any engineering students or an engineer who ca...,any engineering students or an engineer who ca...
352716,352716,481664,481665,How do I take a picture of myself?,What are some good ways to take a picture of m...,1,how do i take a picture of myself,what are some good ways to take a picture of m...,8,11,Art Conservation and Restoration: What are the...,art conservation and restoration what are the...


In [28]:
train = duplicate_data.iloc[:int(duplicate_data.shape[0]*0.80),:]
val = duplicate_data.iloc[int(duplicate_data.shape[0]*0.80):,:]

question1_train = encode_text(train['question1_clean'].tolist(), tokenizer)
question2_train = encode_text(train['question2_clean'].tolist(), tokenizer)
question3_train = encode_text(train['question3_clean'].tolist(), tokenizer)

question1_val = encode_text(val['question1_clean'].tolist(), tokenizer)
question2_val = encode_text(val['question2_clean'].tolist(), tokenizer)
question3_val = encode_text(val['question3_clean'].tolist(), tokenizer)

In [29]:
class DistanceLayer(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, anchor, positive, negative):
        positive_distance = tf.reduce_sum(tf.square(anchor - positive), -1)
        negative_distance = tf.reduce_sum(tf.square(anchor - negative), -1)
        return positive_distance, negative_distance

In [30]:
from tf_keras import metrics

In [31]:
class SiameseModel(Model):

    def __init__(self, siamese_network, margin=0.5):
        super(SiameseModel, self).__init__()
        self.siamese_network = siamese_network
        self.margin = margin
        self.loss_tracker = metrics.Mean(name="loss")

    def call(self, inputs):
        return self.siamese_network(inputs)

    def train_step(self, data):

        with tf.GradientTape() as tape:
            loss = self._compute_loss(data)

        gradients = tape.gradient(loss, self.siamese_network.trainable_weights)
        self.optimizer.apply_gradients(
            zip(gradients, self.siamese_network.trainable_weights)
        )
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        loss = self._compute_loss(data)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def _compute_loss(self, data):
        ap_distance, an_distance = self.siamese_network(data)
        loss = ap_distance - an_distance
        loss = tf.maximum(loss + self.margin, 0.0)
        return loss

    @property
    def metrics(self):
        return [self.loss_tracker]


In [32]:
transformer_model = TFBertModel.from_pretrained(model_checkpoint)

input_ids_in1   = Input(shape=(50,), name='input_ids1',     dtype='int32')
input_masks_in1 = Input(shape=(50,), name='attention_mask1', dtype='int32')

anchor_input   = Input(name='anchor_ids',    shape=(50,), dtype='int32')
anchor_masks   = Input(name='anchor_mask',   shape=(50,), dtype='int32')
positive_input = Input(name='positive_ids',  shape=(50,), dtype='int32')
positive_masks = Input(name='positive_mask', shape=(50,), dtype='int32')
negative_input = Input(name='negative_ids',  shape=(50,), dtype='int32')
negative_masks = Input(name='negative_mask', shape=(50,), dtype='int32')

embedding_layer = transformer_model(input_ids_in1, attention_mask=input_masks_in1).last_hidden_state

average = GlobalAveragePooling1D()(embedding_layer)
embeds  = Dense(512, activation='relu')(average)
embeddings = Model(inputs=[input_ids_in1, input_masks_in1], outputs=embeds)

for layer in embeddings.layers[:-1]:
    layer.trainable = False

embeds1 = embeddings([anchor_input,   anchor_masks])
embeds2 = embeddings([positive_input, positive_masks])
embeds3 = embeddings([negative_input, negative_masks])

distances = DistanceLayer()(embeds1, embeds2, embeds3)

siamese_network = Model(
    inputs=[anchor_input, anchor_masks, positive_input, positive_masks,
            negative_input, negative_masks],
    outputs=distances
)

siamese_model = SiameseModel(siamese_network)
siamese_model.compile(optimizer=tf_keras.optimizers.Adam(0.00001))

# tf.data.Dataset ensures Keras batches along axis-0 (not len(tuple)=6)
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices((
    np.asarray(question1_train['input_ids']),
    np.asarray(question1_train['attention_masks']),
    np.asarray(question2_train['input_ids']),
    np.asarray(question2_train['attention_masks']),
    np.asarray(question3_train['input_ids']),
    np.asarray(question3_train['attention_masks'])
)).batch(BATCH_SIZE)

val_dataset = tf.data.Dataset.from_tensor_slices((
    np.asarray(question1_val['input_ids']),
    np.asarray(question1_val['attention_masks']),
    np.asarray(question2_val['input_ids']),
    np.asarray(question2_val['attention_masks']),
    np.asarray(question3_val['input_ids']),
    np.asarray(question3_val['attention_masks'])
)).batch(BATCH_SIZE)

history = siamese_model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset
)

In [ ]:
plt.figure(figsize=(20,8))
plt.plot(history.history["loss"])
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"])
    plt.legend(["train", "val"], loc="upper left")
else:
    plt.legend(["train"], loc="upper left")
plt.title("model loss")
plt.ylabel("loss")
plt.xlabel("epoch")
plt.show()

In [ ]:
def get_cosine_similarity(sentence1, sentence2):
    f1 = text_cleaning(sentence1)
    f1 = encode_text([f1], tokenizer)
    f2 = text_cleaning(sentence2)
    f2 = encode_text([f2], tokenizer)

    f1_inputs = np.array(f1['input_ids'])
    f1_masks = np.array(f1['attention_masks'])
    f2_inputs = np.array(f2['input_ids'])
    f2_masks = np.array(f2['attention_masks'])

    embeddings1 = embeddings([f1_inputs, f1_masks])
    embeddings2 = embeddings([f2_inputs, f2_masks])

    cosine_similarity = metrics.CosineSimilarity()
    return cosine_similarity(embeddings1, embeddings2).numpy()

In [ ]:
sentence1 = 'Is Earth circle in shape ?'
sentence2 = 'Should I learn python as it is very popular ?'
get_cosine_similarity(sentence1,sentence2)

In [ ]:
sentence1 = 'Python is one of the most popular programming language out there'
sentence2 = 'Should I learn python programming as it is very popular ?'
get_cosine_similarity(sentence1,sentence2)

In [ ]:
sentence1 = 'Which GPU gives a better performance NVIDIA or AMD ?'
sentence2 = 'My friend has a NVIDIA GPU, and he suggests that it gives a very smooth gaming performance'
get_cosine_similarity(sentence1,sentence2)